# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [76]:
# imports
import os
import sys
import requests

sys.path.append(os.path.abspath(".."))
from dotenv import load_dotenv
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
from pathlib import Path
from IPython.display import Markdown, display, update_display



In [65]:
# constants

MODEL_GPT = 'gpt-5-nano'
MODEL_LLAMA = 'llama3.2'

In [87]:
# set up environment
load_dotenv(Path("../../../.env"))
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")
openai = OpenAI()

API key found and looks good so far!


In [88]:
system_prompt = """
You are a knowledgeable programming tutor.

Your job is to help students understand technical questions and code.
Always provide clear, step-by-step explanations.

Respond in Markdown format.
Do NOT wrap the entire response in a Markdown code block.
"""

In [43]:
def gen_user_prompt(question):
    user_prompt = f"""
Here is the technical question I need help with:

{question}

Please explain it clearly and step-by-step.
"""
    return user_prompt

In [48]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:

def count_up_to(n):
    i = 1
    while i <= n:
        yield i
        i += 1
"""

In [66]:
# Get gpt-5-nano to answer, with streaming
def answer_question_gpt(question):
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages = [
            {"role":"system", "content": system_prompt},
            {"role":"user", "content": gen_user_prompt(question)}
        ],
        stream=True
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [84]:
# requests.get("http://localhost:11434").content

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [85]:
# Get Llama 3.2 to answer
def answer_question_llama(question):
    stream = ollama.chat.completions.create(
        model=MODEL_LLAMA,
        messages = [
            {"role":"system", "content": system_prompt},
            {"role":"user", "content": gen_user_prompt(question)}
        ],
        stream=True
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [86]:
answer_question_llama(question)

# Understanding the `count_up_to` Function
=====================================

The `count_up_to` function is a generator that yields numbers from 1 up to, but not including, the given number `n`.

### Step-by-Step Explanation
-------------------------

Here's how it works:

1. **Initialization**: We initialize a counter variable `i` to 1.
2. **Loop Condition**: The loop condition checks whether `i` is less than or equal to `n`. As long as this condition is true, the loop will continue.
3. **Yielding a Value**: Inside the loop, we use the `yield` keyword to produce a value, which in this case is the current value of `i`.
4. **Incrementing `i`**: After yielding a value, we increment `i` by 1 to prepare it for the next iteration.

### Key Points
------------

*   The `count_up_to` function returns an iterator that produces numbers on-the-fly, rather than all at once.
*   It does not store any of these values in memory; instead, it generates them as needed when requested.
*   Once the loop condition is no longer true (i.e., after `n` has been reached), there are no more values to yield.

### Example Use Case
-------------------

Here's an example of how you might use this function:
```python
for num in count_up_to(5):
    print(num)
```

Output:
```
1
2
3
4
5
```
As the loop iterates and yields numbers, it skips over any value that would cause the condition (`i <= n`) to become false.